# Stage 2 — Cleaning

Null handling, deduplication, date standardisation, skill list parsing.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src.utils import (
    load_csv, standardise_columns, drop_null_rows,
    fill_nulls, fill_nulls_grouped, remove_duplicates,
    standardise_dates, parse_skill_list,
)
import pandas as pd

In [ ]:
df_jobs = load_csv("data/raw/sample_job_postings.csv", parse_dates=["date_posted"])
df_placements = load_csv("data/raw/placement_outcomes.csv", parse_dates=["placement_date"])

## Standardise column names

In [ ]:
df_jobs = standardise_columns(df_jobs)
df_placements = standardise_columns(df_placements)
print(df_jobs.columns.tolist())

## Drop near-empty rows

In [ ]:
df_jobs = drop_null_rows(df_jobs, threshold=0.5)
df_placements = drop_null_rows(df_placements, threshold=0.5)
print(f"After drop_null_rows — jobs: {df_jobs.shape}, placements: {df_placements.shape}")

## Fill nulls (salary by sector-median)

In [ ]:
if "sector" in df_jobs.columns:
    df_jobs = fill_nulls_grouped(df_jobs, col="salary_lpa", group_col="sector")
df_placements = fill_nulls(df_placements, strategy="median")
print("Null counts after fill:
", df_jobs.isnull().sum()[df_jobs.isnull().sum() > 0])

## Remove duplicates

In [ ]:
df_jobs = remove_duplicates(df_jobs, subset=["job_id"])
df_placements = remove_duplicates(df_placements, subset=["candidate_id"])

## Parse skill lists

In [ ]:
df_jobs["skills_list"] = df_jobs["skills_required"].apply(parse_skill_list)
df_placements["skills_list"] = df_placements["skills"].apply(parse_skill_list)
df_jobs[["job_id", "skills_list"]].head()

## Save interim checkpoints

In [ ]:
df_jobs.to_csv("data/interim/jobs_cleaned.csv", index=False)
df_placements.to_csv("data/interim/placements_cleaned.csv", index=False)
print("Saved to data/interim/")